# 06 - Candidate segmentations (DS, Day 9)

The DA plan's Day 9 task for DS: **shortlist 2 to 3 candidate segmentations and share the labels with the DA team.**

The baseline (notebook 05) found two things this notebook acts on:
1. **One cluster (5.6%) was not a customer type.** It held customers with no web sessions: the `has_sessions` / `has_views` flags and the "zero sessions" counts were enough to pull them together. Here, customers with no browsing data get the *typical* browsing values and no flag, so they are placed by what we do know about them (profile and enquiries).
2. **One-hot categories carry a lot of weight in k-means** (84 of 144 columns). Here, each categorical variable is weighted so it counts about as much as one numeric feature.

**Candidates**

| | Model | Why |
|---|---|---|
| A | k-means on the fixed, re-weighted matrix | The baseline, corrected |
| B | Gaussian mixture on principal components | Allows clusters of different shapes and sizes |
| C | k-prototypes (fitted on a sample, then applied to everyone) | Handles numbers and categories natively |
| D | Two-stage k-means: split by intent (Buy / Rent), then cluster each group | Buying and renting are different markets, so they should not share clusters |

**Rules:** no answer key (`_restricted/`) and no outcomes. Candidates are compared on stability, separation, agreement with each other, and whether the clusters make business sense. The README's ARI target is checked only after modelling is final.

## 0. Setup

In [1]:
%pip install -q scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install -q kmodes

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from pandas.api.types import is_numeric_dtype

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent
FEAT = ROOT / "data" / "features"
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(exist_ok=True)
for f in ["customer_features_mixed.parquet", "customer_features.parquet", "labels_kmeans_baseline.parquet"]:
    assert (FEAT / f).exists(), f"{f} missing: run notebooks 04 and 05 first"

X = pd.read_parquet(FEAT / "customer_features_mixed.parquet").set_index("customer_id")
F = pd.read_parquet(FEAT / "customer_features.parquet").set_index("customer_id").loc[X.index]
base = pd.read_parquet(FEAT / "labels_kmeans_baseline.parquet").set_index("customer_id").iloc[:, 0].loc[X.index]

cat_cols = [c for c in X.columns if not is_numeric_dtype(X[c])]
num_cols = [c for c in X.columns if c not in cat_cols]
K = 8                                   # CONFIG: number of clusters (see notebook 05 for why 8)
print(X.shape, f"| {len(num_cols)} numeric, {len(cat_cols)} categorical | K = {K}")

(100573, 70) | 60 numeric, 10 categorical | K = 8


## 1. Fix the data-availability problem
About 5% of customers have no identified web session, so we know nothing about their browsing. Saying "zero sessions" is not the same as "unknown".
- `has_sessions` and `has_views` are removed from the model.
- For those customers, session and view counts, pages and duration take the **median of the customers who do have data**. (The shares were already filled with typical values in notebook 04.)
- They are therefore clustered on profile and enquiry features only.

In [4]:
no_sess = F["has_sessions"].eq(0)
no_view = F["has_views"].eq(0)
print(f"customers with no sessions: {no_sess.sum()} ({no_sess.mean():.1%}) | no real views: {no_view.sum()} ({no_view.mean():.1%})")

Xn = X[num_cols].drop(columns=["has_sessions", "has_views"], errors="ignore").copy()
for c in ["n_sessions", "mean_pages", "mean_duration_min"]:
    if c in Xn:
        Xn.loc[no_sess, c] = Xn.loc[~no_sess, c].median()
if "n_views" in Xn:
    Xn.loc[no_view, "n_views"] = Xn.loc[~no_view, "n_views"].median()

W_CAT = 0.7071      # a mismatch on a one-hot variable contributes 1 to the squared distance, like a 1 SD gap on a numeric feature
Xc = pd.get_dummies(X[cat_cols], dtype=float) * W_CAT
M = pd.concat([Xn, Xc], axis=1)
print(f"Model matrix: {M.shape[1]} columns ({Xn.shape[1]} numeric + {Xc.shape[1]} weighted category columns)")

customers with no sessions: 4996 (5.0%) | no real views: 5472 (5.4%)
Model matrix: 142 columns (58 numeric + 84 weighted category columns)


## 2. Candidate A: k-means, fixed
Same method as the baseline, on the corrected and re-weighted matrix.

In [5]:
t0 = time.time()
km = KMeans(n_clusters=K, n_init=20, random_state=0).fit(M)
labA = pd.Series(km.labels_, index=M.index, name="A_kmeans_fixed")
print(f"A fitted in {time.time()-t0:.0f}s | inertia {km.inertia_:,.0f}")
labA.value_counts(normalize=True).sort_index().mul(100).round(1).to_frame("share_%")

A fitted in 9s | inertia 4,081,862


,share_%
A_kmeans_fixed,
0,2.8
1,27.3
2,6.4
3,6.8
4,11.8
5,22.2
6,17.3
7,5.4


## 3. Candidate B: Gaussian mixture on principal components
A mixture model fits clusters that can differ in size and spread. It runs on the first principal components (the directions that carry most of the variation) to keep it fast and stable, then each customer goes to its most likely cluster.

In [6]:
t0 = time.time()
pca = PCA(n_components=0.90, random_state=0).fit(M)
Z = pca.transform(M)
print(f"PCA keeps {Z.shape[1]} components for {pca.explained_variance_ratio_.sum():.0%} of the variance")
gm = GaussianMixture(n_components=K, covariance_type="full", n_init=2, max_iter=200, random_state=0).fit(Z)
labB = pd.Series(gm.predict(Z), index=M.index, name="B_gmm")
print(f"B fitted in {time.time()-t0:.0f}s | converged: {gm.converged_}")
labB.value_counts(normalize=True).sort_index().mul(100).round(1).to_frame("share_%")

PCA keeps 39 components for 90% of the variance
B fitted in 50s | converged: True


,share_%
B_gmm,
0,17.7
1,29.2
2,12.0
3,14.5
4,10.3
5,5.2
6,4.9
7,6.1


## 4. Candidate C: k-prototypes (optional, slower)
k-prototypes uses distance for the numbers and *matching* for the categories, so no weighting trick is needed. It is slow, so it is **fitted on a random sample** and then applied to every customer.
Set `RUN_KPROTOTYPES = False` to skip it. If the package is missing, the cell says so and carries on.

In [7]:
RUN_KPROTOTYPES = True
SAMPLE_N = 15_000
labC = None
if RUN_KPROTOTYPES:
    try:
        from kmodes.kprototypes import KPrototypes
        t0 = time.time()
        data = pd.concat([Xn, X[cat_cols].astype(str)], axis=1)
        cat_idx = list(range(Xn.shape[1], data.shape[1]))
        arr = data.to_numpy(dtype=object)
        rng = np.random.default_rng(0)
        sample = rng.choice(len(arr), size=min(SAMPLE_N, len(arr)), replace=False)
        kp = KPrototypes(n_clusters=K, init="Cao", n_init=1, max_iter=15, random_state=0)
        kp.fit(arr[sample], categorical=cat_idx)
        print(f"C fitted on {len(sample)} customers in {time.time()-t0:.0f}s; now assigning everyone ...")
        pred = np.concatenate([kp.predict(arr[i:i + 10_000], categorical=cat_idx) for i in range(0, len(arr), 10_000)])
        labC = pd.Series(pred, index=M.index, name="C_kprototypes")
        print(f"C done in {time.time()-t0:.0f}s")
    except ImportError:
        print("kmodes is not installed, so candidate C is skipped. Run:  %pip install kmodes")
labC.value_counts(normalize=True).sort_index().mul(100).round(1).to_frame("share_%") if labC is not None else "skipped"

C fitted on 15000 customers in 12s; now assigning everyone ...
C done in 15s


,share_%
C_kprototypes,
0,20.8
1,16.8
2,17.6
3,16.4
4,6.8
5,6.3
6,12.5
7,2.8


## 4b. Candidate D: two-stage k-means (split by intent first)
Notebook 05 showed renters forming their own clean cluster, but candidate A (with categories down-weighted) lets renters and buyers mix. Intent changes the whole picture: rent is a monthly figure, a purchase is a lump sum, and the listings viewed differ.
So D first splits customers by `primary_intent`, then runs k-means separately inside each group. `K_RENT` clusters go to renters and the rest to buyers.

In [8]:
K_RENT = 2                      # CONFIG: renters are about 21% of customers, so 2 clusters is a starting point
K_BUY = K - K_RENT

def two_stage(seed, n_init):
    lab = pd.Series(index=M.index, dtype=object)
    for intent, k, tag in [("Rent", K_RENT, "R"), ("Buy", K_BUY, "B")]:
        idx = X.index[X["primary_intent"].eq(intent)]
        part = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit_predict(M.loc[idx])
        lab.loc[idx] = [f"{tag}{c}" for c in part]
    return lab

t0 = time.time()
labD = two_stage(0, 10).rename("D_two_stage")
print(f"D fitted in {time.time()-t0:.0f}s | renters: {int(X['primary_intent'].eq('Rent').sum())}, buyers: {int(X['primary_intent'].eq('Buy').sum())}, other: {int((~X['primary_intent'].isin(['Rent','Buy'])).sum())}")
assert labD.notna().all(), "Some customers have an intent other than Buy or Rent: handle them before clustering"
labD.value_counts(normalize=True).sort_index().mul(100).round(1).to_frame("share_%")

D fitted in 3s | renters: 20885, buyers: 79688, other: 0


,share_%
D_two_stage,
B0,6.5
B1,17.4
B2,14.2
B3,22.2
B4,12.1
B5,6.8
R0,3.1
R1,17.7


## 5. Compare the candidates
Four views, none of which uses the answer key:
- **Size:** the smallest cluster. Tiny clusters are usually noise.
- **Separation:** silhouette on a 15,000-customer sample, in the same matrix for every candidate (low values are normal here).
- **Stability:** re-fit with different random starts and compare with the main run (adjusted Rand index; 1 = identical).
- **Agreement:** do different methods find the same groups? (ARI between candidates). High agreement means the structure is real, not an artefact of one method.

In [9]:
cands = {"baseline (05)": base, "A": labA, "B": labB}
if labC is not None:
    cands["C"] = labC
cands["D"] = labD
cands = {k: v.rename(k) for k, v in cands.items()}

sil_idx = np.random.default_rng(0).choice(len(M), size=min(15_000, len(M)), replace=False)
rows = []
for name, lab in cands.items():
    sh = lab.value_counts(normalize=True) * 100
    rows.append({"candidate": name, "clusters": lab.nunique(), "smallest_%": round(sh.min(), 1), "largest_%": round(sh.max(), 1),
                 "silhouette": round(silhouette_score(M.iloc[sil_idx], lab.iloc[sil_idx]), 3)})
summary = pd.DataFrame(rows).set_index("candidate")

stab = {}
stab["A"] = np.mean([adjusted_rand_score(labA, KMeans(n_clusters=K, n_init=3, random_state=s).fit_predict(M)) for s in range(1, 6)])
stab["B"] = np.mean([adjusted_rand_score(labB, GaussianMixture(n_components=K, covariance_type="full", n_init=1, max_iter=200, random_state=s).fit(Z).predict(Z)) for s in range(1, 5)])
stab["D"] = np.mean([adjusted_rand_score(labD, two_stage(s_, 3)) for s_ in range(1, 4)])
summary["stability_ARI"] = pd.Series(stab).round(3)
display(summary)

agree = pd.DataFrame({a: {b: adjusted_rand_score(cands[a], cands[b]) for b in cands} for a in cands}).round(2)
print("Agreement between candidates (ARI):")
display(agree)

,clusters,smallest_%,largest_%,silhouette,stability_ARI
candidate,,,,,
baseline (05),8,5.6,21.1,0.031,NaN
A,8,2.8,27.3,0.070,0.645
B,8,4.9,29.2,0.037,0.609
C,8,2.8,20.8,0.046,NaN
D,8,3.1,22.2,0.045,0.943


Agreement between candidates (ARI):


,baseline (05),A,B,C,D
baseline (05),1.00,0.48,0.26,0.83,0.87
A,0.48,1.00,0.42,0.56,0.53
B,0.26,0.42,1.00,0.28,0.27
C,0.83,0.56,0.28,1.00,0.86
D,0.87,0.53,0.27,0.86,1.00


## 6. What is in each candidate?
Plain-unit profiles for every candidate, using the readable features. Read them next to each other: the same customer type should show up in each, and each cluster should be explainable in a sentence.

In [10]:
def profile(lab):
    p = pd.DataFrame(index=sorted(lab.unique()))
    p["customers"] = lab.value_counts().sort_index()
    p["share_%"] = (p["customers"] / len(lab) * 100).round(1)
    p["buy_%"] = (F["primary_intent"].eq("Buy").groupby(lab).mean() * 100).round(0)
    p["overseas_%"] = (F["is_overseas"].astype(float).groupby(lab).mean() * 100).round(0)
    p["median_budget"] = F["budget_gbp"].groupby(lab).median().round(0)
    p["budget_missing_%"] = (F["budget_missing"].groupby(lab).mean() * 100).round(0)
    for c, n in [("share_mobile", "mobile_%"), ("share_new_build", "new_build_%"), ("share_offplan", "offplan_%")]:
        if c in F.columns:
            p[n] = (F[c].groupby(lab).mean() * 100).round(0)
    p["median_sessions"] = F["n_sessions"].groupby(lab).median()
    p["no_sessions_%"] = ((F["has_sessions"] == 0).groupby(lab).mean() * 100).round(0)
    p["top_buyer_position"] = F["buyer_position"].fillna("Unknown").groupby(lab).agg(lambda s: s.value_counts().index[0])
    p["top_finance"] = F["finance_type"].fillna("Unknown").groupby(lab).agg(lambda s: s.value_counts().index[0])
    p["top_lead_source"] = F["primary_lead_source"].fillna("No lead").groupby(lab).agg(lambda s: s.value_counts().index[0])
    return p

for name, lab in cands.items():
    print(f"\n=== {name} ===")
    display(profile(lab))


=== baseline (05) ===


,customers,share_%,buy_%,overseas_%,median_budget,budget_missing_%,mobile_%,new_build_%,offplan_%,median_sessions,no_sessions_%,top_buyer_position,top_finance,top_lead_source
0,12120,12.1,99.0,3.0,340000.0,68.0,61.0,31.0,11.0,32.0,0.0,Not stated,Mortgage,No lead
1,21190,21.1,99.0,0.0,235000.0,20.0,81.0,32.0,9.0,12.0,0.0,First-time buyer,Mortgage,No lead
2,13268,13.2,88.0,7.0,180000.0,38.0,27.0,23.0,7.0,8.0,0.0,Portfolio landlord,Buy-to-let mortgage,Email newsletter
3,7780,7.7,88.0,90.0,536250.0,56.0,53.0,81.0,45.0,11.0,0.0,Overseas investor,Cash,No lead
4,17925,17.8,98.0,2.0,460000.0,31.0,50.0,19.0,4.0,8.0,0.0,Home mover (in chain),Mortgage,No lead
5,5654,5.6,71.0,9.0,305000.0,38.0,59.0,29.0,4.0,0.0,88.0,Downsizer (chain-free),Mortgage,No lead
6,6335,6.3,100.0,2.0,385000.0,45.0,27.0,21.0,5.0,4.0,0.0,Downsizer (chain-free),Cash,No lead
7,16301,16.2,2.0,1.0,1400.0,34.0,82.0,19.0,0.0,9.0,0.0,Renter,Renting,No lead



=== A ===


,customers,share_%,buy_%,overseas_%,median_budget,budget_missing_%,mobile_%,new_build_%,offplan_%,median_sessions,no_sessions_%,top_buyer_position,top_finance,top_lead_source
0,2807,2.8,3.0,37.0,2900.0,59.0,33.0,38.0,1.0,3.0,0.0,Corporate tenant,Company let,No lead
1,27492,27.3,71.0,3.0,247500.0,47.0,72.0,29.0,7.0,15.0,12.0,First-time buyer,Mortgage,No lead
2,6440,6.4,100.0,2.0,385000.0,45.0,27.0,21.0,5.0,4.0,2.0,Downsizer (chain-free),Cash,No lead
3,6836,6.8,99.0,92.0,580000.0,56.0,55.0,86.0,52.0,13.0,0.0,Overseas investor,Cash,No lead
4,11883,11.8,96.0,7.0,190000.0,36.0,27.0,22.0,8.0,8.0,4.0,Portfolio landlord,Buy-to-let mortgage,Email newsletter
5,22284,22.2,60.0,1.0,185000.0,24.0,81.0,28.0,5.0,9.0,5.0,First-time buyer,Renting,Rightmove
6,17392,17.3,99.0,1.0,470000.0,31.0,50.0,19.0,4.0,8.0,0.0,Home mover (in chain),Mortgage,No lead
7,5439,5.4,88.0,1.0,212500.0,32.0,67.0,21.0,7.0,12.0,0.0,First-time buyer,Mortgage,No lead



=== B ===


,customers,share_%,buy_%,overseas_%,median_budget,budget_missing_%,mobile_%,new_build_%,offplan_%,median_sessions,no_sessions_%,top_buyer_position,top_finance,top_lead_source
0,17850,17.7,66.0,0.0,190000.0,25.0,80.0,30.0,6.0,12.0,0.0,First-time buyer,Renting,Rightmove
1,29397,29.2,76.0,2.0,280000.0,44.0,67.0,26.0,7.0,14.0,5.0,First-time buyer,Mortgage,No lead
2,12100,12.0,95.0,3.0,350000.0,39.0,39.0,22.0,5.0,3.0,0.0,Downsizer (chain-free),Cash,No lead
3,14535,14.5,86.0,1.0,385000.0,31.0,60.0,20.0,4.0,9.0,11.0,Home mover (in chain),Mortgage,Rightmove
4,10358,10.3,93.0,8.0,185000.0,36.0,28.0,22.0,8.0,8.0,18.0,Portfolio landlord,Buy-to-let mortgage,Email newsletter
5,5280,5.2,39.0,35.0,3850.0,52.0,41.0,41.0,9.0,2.0,0.0,Corporate tenant,Company let,No lead
6,4962,4.9,76.0,2.0,210000.0,33.0,65.0,19.0,5.0,12.0,0.0,First-time buyer,Mortgage,No lead
7,6091,6.1,99.0,91.0,585000.0,55.0,55.0,85.0,51.0,15.0,0.0,Overseas investor,Cash,No lead



=== C ===


,customers,share_%,buy_%,overseas_%,median_budget,budget_missing_%,mobile_%,new_build_%,offplan_%,median_sessions,no_sessions_%,top_buyer_position,top_finance,top_lead_source
0,20893,20.8,97.0,0.0,232500.0,21.0,81.0,32.0,8.0,11.0,3.0,First-time buyer,Mortgage,No lead
1,16891,16.8,10.0,1.0,1450.0,31.0,82.0,20.0,1.0,8.0,4.0,Renter,Renting,No lead
2,17741,17.6,99.0,1.0,460000.0,31.0,50.0,19.0,4.0,8.0,0.0,Home mover (in chain),Mortgage,No lead
3,16537,16.4,90.0,4.0,305000.0,62.0,62.0,31.0,11.0,24.0,19.0,Not stated,Mortgage,No lead
4,6819,6.8,99.0,92.0,580000.0,56.0,55.0,86.0,52.0,13.0,0.0,Overseas investor,Cash,No lead
5,6324,6.3,100.0,2.0,385000.0,45.0,26.0,21.0,5.0,4.0,1.0,Downsizer (chain-free),Cash,No lead
6,12573,12.5,96.0,7.0,190000.0,36.0,26.0,22.0,8.0,8.0,5.0,Portfolio landlord,Buy-to-let mortgage,Email newsletter
7,2795,2.8,4.0,37.0,2900.0,59.0,33.0,38.0,1.0,3.0,0.0,Corporate tenant,Company let,No lead



=== D ===


,customers,share_%,buy_%,overseas_%,median_budget,budget_missing_%,mobile_%,new_build_%,offplan_%,median_sessions,no_sessions_%,top_buyer_position,top_finance,top_lead_source
B0,6503,6.5,100.0,2.0,382500.0,45.0,27.0,21.0,5.0,4.0,2.0,Downsizer (chain-free),Cash,No lead
B1,17530,17.4,100.0,1.0,462500.0,31.0,50.0,19.0,4.0,8.0,0.0,Home mover (in chain),Mortgage,No lead
B2,14310,14.2,100.0,3.0,335000.0,63.0,61.0,32.0,11.0,25.0,14.0,First-time buyer,Mortgage,No lead
B3,22337,22.2,100.0,0.0,240000.0,21.0,81.0,32.0,9.0,11.0,5.0,First-time buyer,Mortgage,No lead
B4,12178,12.1,100.0,6.0,190000.0,35.0,27.0,21.0,8.0,8.0,3.0,Portfolio landlord,Buy-to-let mortgage,Email newsletter
B5,6830,6.8,100.0,92.0,585000.0,56.0,55.0,86.0,52.0,13.0,0.0,Overseas investor,Cash,No lead
R0,3101,3.1,0.0,35.0,2800.0,60.0,33.0,37.0,0.0,3.0,1.0,Corporate tenant,Company let,No lead
R1,17784,17.7,0.0,3.0,1400.0,35.0,82.0,19.0,0.0,8.0,8.0,Renter,Renting,No lead


## 7. Shortlist and save
Choose 2 or 3 to share with the DA team. Decide on **stability, size, agreement and whether each cluster makes sense**, and write the reason down.
The DA team will profile the shortlisted candidates on Day 10-11, including against outcomes, which this notebook never touches.

In [11]:
SHORTLIST = [k for k in cands if k != "baseline (05)"]      # CONFIG: trim to the 2 or 3 you choose after reading sections 5 and 6

out = pd.concat([cands[k] for k in cands], axis=1)
out.reset_index().to_parquet(FEAT / "labels_candidates.parquet", index=False)
summary.to_csv(OUT / "candidate_comparison.csv")
agree.to_csv(OUT / "candidate_agreement.csv")
for k in SHORTLIST:
    profile(cands[k]).to_csv(OUT / f"candidate_{k}_profile.csv")
print("Saved labels for", list(cands), "->", FEAT / "labels_candidates.parquet")
print("Shortlist:", SHORTLIST)

Saved labels for ['baseline (05)', 'A', 'B', 'C', 'D'] -> c:\Users\pc\Documents\AMDARI INTERNSHIP\Veltrix Segmentation\veltrix-segmentation_A\data\features\labels_candidates.parquet
Shortlist: ['A', 'B', 'C', 'D']


## 8. Validate

In [12]:
results = {
    "every customer has a label in every candidate": bool(out.notna().all().all()) and len(out) == len(X),
    "no candidate has a cluster under 1% of customers": bool((summary.loc[[c for c in summary.index if c != 'baseline (05)'], 'smallest_%'] >= 1).all()),
    "no cluster is defined by missing browsing data (A)": bool(profile(labA)["no_sessions_%"].max() < 50),
    "candidate A is stable (ARI >= 0.7)": bool(stab["A"] >= 0.7),
    "answer key never read": True,
}
for k_, v in results.items():
    print(("PASS  " if v else "CHECK ") + k_)
print("\n'CHECK' is something to look at and write down, not an error.")

PASS  every customer has a label in every candidate
PASS  no candidate has a cluster under 1% of customers
PASS  no cluster is defined by missing browsing data (A)
CHECK candidate A is stable (ARI >= 0.7)
PASS  answer key never read

'CHECK' is something to look at and write down, not an error.


In [ ]:
for k in ["B", "C", "D"]:
    print(f"=== {k} ===")
    display(profile(cands[k]).T)

## Hand-off notes
- **Share with DA (Day 9):** `data/features/labels_candidates.parquet` and `outputs/candidate_comparison.csv`, plus the profile CSVs for the shortlist.
- **Write down:** which candidates you shortlisted and why, the stability numbers, and the baseline's no-sessions cluster and how this notebook handled it.
- **Day 10:** stability checks on the final choice (bootstrap, not just random starts), then write `customer_segments` with `model_version = seg-v1.0`.
- **Only when the model is final:** run `recoverability_check.py` once against the answer key, and report the ARI as a final result. Do not tune against it.